In [2]:
import sounddevice as sd
import numpy as np

# Updated for DeepFilterNet3 requirements
SAMPLE_RATE = 48000  
# 960 samples = exactly 20ms at 48kHz (perfect low-latency UI buffer size)
BLOCK_SIZE = 960      

def audio_callback(indata, outdata, frames, time_info, status):
    if status:
        print(f"\nStatus flag: {status}")
    
    # Extract channels
    # If using laptop mic, PipeWire usually sends the mono mic to both channels natively
    mic1_voice = indata[:, 0]  
    mic2_noise = indata[:, 1]  

    # Compute RMS levels for visual telemetry
    rms_voice = np.sqrt(np.mean(mic1_voice**2)) + 1e-9
    rms_noise = np.sqrt(np.mean(mic2_noise**2)) + 1e-9

    # Convert to approximate dBFS
    db_voice = 20 * np.log10(rms_voice)
    db_noise = 20 * np.log10(rms_noise)

    # Simple terminal meter (updates inline)
    meter = f"\r[Laptop Mic]: {db_voice:6.1f} dBFS | [CH2/Dummy]: {db_noise:6.1f} dBFS"
    print(meter, end="", flush=True)

    # Pass-through test: Duplicate mic1 to both Focusrite headphone ears
    outdata[:, 0] = mic1_voice
    outdata[:, 1] = mic1_voice

print(f"Starting 48kHz audio stream. Speak into laptop mic...")
print("You should hear yourself in the Focusrite headphones.")
with sd.Stream(channels=(2, 2), samplerate=SAMPLE_RATE, blocksize=BLOCK_SIZE, dtype='float32', callback=audio_callback):
    input("\nPress Enter to stop stream.\n")

Starting 48kHz audio stream. Speak into laptop mic...
You should hear yourself in the Focusrite headphones.
[Laptop Mic]: -100.9 dBFS | [CH2/Dummy]:  -42.4 dBFS
Status flag: output underflow
[Laptop Mic]:  -28.3 dBFS | [CH2/Dummy]:  -23.2 dBFS
Press Enter to stop stream.

[Laptop Mic]:  -27.5 dBFS | [CH2/Dummy]:  -24.4 dBFS

In [5]:
import sounddevice as sd
import numpy as np

# Let PipeWire handle the hardware bridging
SAMPLE_RATE = 48000  
BLOCK_SIZE = 960      

def audio_callback(indata, outdata, frames, time_info, status):
    mic_voice = indata[:, 0]  
    db = 20 * np.log10(np.sqrt(np.mean(mic_voice**2)) + 1e-9)
    print(f"\r[Streaming via PipeWire]: {db:6.1f} dBFS ", end="", flush=True)

    outdata[:, 0] = mic_voice
    outdata[:, 1] = mic_voice

print("Starting audio stream... Leave this running.")
with sd.Stream(channels=(1, 2), samplerate=SAMPLE_RATE, blocksize=BLOCK_SIZE, dtype='float32', callback=audio_callback):
    input("\nPress Enter to stop stream.\n")

Starting audio stream... Leave this running.
[Streaming via PipeWire]:  -25.8 dBFS 
Press Enter to stop stream.

[Streaming via PipeWire]:  -25.6 dBFS 